# 🚀 Fine-Tuning Qwen3.5 2B for Indian SMS Entity Extraction

**Production-Aligned Interactive Training, Evaluation & Inference Notebook**

This notebook mirrors the production `slm_sms` package (`src/slm_sms/`) end-to-end:
1. **Centralized Config & Observability** (`slm_sms.core`: YAML configs, `settings`, JSON/text logging, OpenTelemetry)
2. **Indian SMS Corpus Synthesis** (`IndianSMSGenerator` with thread-safe RNG, Indian lakh/crore grouping, class-balance guard)
3. **Base Model Inspection & LoRA Configuration** (`slm_sms.models.model_loader`)
4. **Supervised Fine-Tuning** (`SMSTrainingPipeline` with TRL `SFTTrainer` & PEFT LoRA)
5. **Test-Set Benchmarking** (`compute_extraction_metrics` over `data/test.jsonl`)
6. **Single & Batched Inference** (`SMSExtractorEngine.extract` & `extract_batch` with automatic regex fallback)
7. **FastAPI Microservice & OpenTelemetry Verification** (`create_app`, health probes, `X-Request-ID`, `X-Trace-ID`, spans)

### 🛠️ Architecture
```mermaid
flowchart LR
    A["Raw Indian SMS"] --> B["IndianSMSGenerator"]
    B --> C["Chat JSONL Splits (train / val / test)"]
    C --> D["SMSTrainingPipeline (Qwen3.5-2B + LoRA)"]
    D --> E["Trained LoRA Adapter (output/qwen-sms-lora)"]
    E --> F["SMSExtractorEngine (Single + GPU Batch + Fallback)"]
    F --> G["FastAPI + Gunicorn + OpenTelemetry"]
```


## 1. Setup, Configuration, Logging, Telemetry & Hardware Diagnostics
Initialize `slm_sms.core` logging, load `config/generation_config.yaml` and `config/train_config.yaml`, and inspect the available compute device (CUDA / Apple Silicon MPS / CPU).

In [1]:
import gc
import os
import sys
from pathlib import Path

# Ensure project src/ is importable when launched directly from notebooks/
project_root = Path("..").resolve()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import datasets
import peft
import pydantic
import torch
import transformers
import trl

from slm_sms.core import configure_logging, configure_telemetry, load_yaml_config, settings
from slm_sms.models.model_loader import get_device_and_dtype

# Readable logs inside Jupyter; OpenTelemetry spans buffered in-memory until configured
configure_logging(level="INFO", use_json=False)
configure_telemetry(exporter_mode="none")

# Load production YAML configs
gen_cfg = load_yaml_config(project_root / "config" / "generation_config.yaml")
train_cfg = load_yaml_config(project_root / "config" / "train_config.yaml")

MODEL_ID = train_cfg["model"]["model_id"]  # Default: Qwen/Qwen3.5-2B
MODEL_REVISION = settings.model_revision  # Optional commit hash pin (SLM_MODEL_REVISION)
DATA_DIR = project_root / "data"
OUTPUT_DIR = project_root / "output" / "qwen-sms-lora-notebook"

device, dtype, dev_name = get_device_and_dtype()
print(f"Python:       {sys.version.split()[0]}")
print(f"PyTorch:      {torch.__version__} | Transformers: {transformers.__version__}")
print(f"PEFT:         {peft.__version__} | TRL: {trl.__version__}")
print(f"Datasets:     {datasets.__version__} | Pydantic: {pydantic.__version__}")
print(f"Device:       {dev_name} ({device}, dtype={dtype})")
if dev_name == "cuda":
    print(f"GPU Name:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Target Model: {MODEL_ID} (revision={MODEL_REVISION or 'latest'})")

W1125 20:12:09.156000 4064 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


2026-11-25T20:12:09 | INFO     | slm_sms.core.telemetry | OpenTelemetry configured
Python:       3.12.12
PyTorch:      2.15.0 | Transformers: 5.24.0
PEFT:         0.22.0 | TRL: 1.18.2
Datasets:     5.2.0 | Pydantic: 2.14.4
Device:       mps (mps, dtype=torch.float16)
Target Model: Qwen/Qwen3.5-2B (revision=latest)


## 2. Indian SMS Dataset Generation & Exploration
Uses [`IndianSMSGenerator`](../src/slm_sms/data/generator.py) driven by `config/generation_config.yaml`:
- **Thread-safe seeding**: `random.Random(seed)` isolated per generator instance
- **Indian numbering system**: `_format_indian` formats lakh/crore amounts (`1,45,000.00`, `14,50,000.00`)
- **Class imbalance guard**: Logs class distribution and raises if any SMS class drops below `min_class_fraction` (`2%`)

In [2]:
import json
from collections import Counter

from slm_sms.data.dataset import save_jsonl
from slm_sms.data.generator import IndianSMSGenerator

ds_params = gen_cfg["dataset"]
generator = IndianSMSGenerator(seed=int(ds_params["seed"]))

data_splits = generator.generate_dataset(
    n_samples=int(ds_params["samples"]),
    val_split=float(ds_params["val_split"]),
    test_split=float(ds_params["test_split"]),
)

print(f"Train split:      {len(data_splits['train'])} samples")
print(f"Validation split: {len(data_splits['validation'])} samples")
print(f"Test split:       {len(data_splits['test'])} samples")

save_jsonl(data_splits["train"], DATA_DIR / "train.jsonl")
save_jsonl(data_splits["validation"], DATA_DIR / "val.jsonl")
save_jsonl(data_splits["test"], DATA_DIR / "test.jsonl")
print(f"Saved JSONL splits to {DATA_DIR}/")

# Inspect class distribution
train_types = Counter(
    json.loads(item["messages"][2]["content"])["type"] for item in data_splits["train"]
)
print("Train type distribution:", dict(train_types))

2026-11-25T20:12:09 | INFO     | slm_sms.data.generator | Dataset class distribution
Train split:      2000 samples
Validation split: 375 samples
Test split:       125 samples
Saved JSONL splits to /Users/parthsojitra/Work/Projects/SLMTraining/data/
Train type distribution: {'credit': 301, 'promo': 45, 'otp': 292, 'debit': 992, 'bill': 178, 'spam': 98, 'refund': 94}


### Inspect Sample Chat Messages & Indian Lakh Formatting

In [3]:
print("Indian number format check: Rs.", generator._format_indian(145000.0))
print()

for i in range(4):
    sample = data_splits["train"][i]["messages"]
    print(f"=== Sample {i + 1} ===")
    print(f"System:    {sample[0]['content']}")
    print(f"User SMS:  {sample[1]['content']}")
    print(f"Assistant: {sample[2]['content']}")
    print()

Indian number format check: Rs. 1,45,000.00

=== Sample 1 ===
System:    Extract structured JSON from SMS. Output valid JSON only.
User SMS:  [AD-AXISBK] A/c *2402 credited by INR 1,13,957.32 on 21/10/26 by Salary transfer from GOOGLE INDIA. Avl Bal: INR 46,406.98.
Assistant: {"type":"credit","amount":113957.32,"currency":"INR","merchant":"GOOGLE INDIA","account":"2402","balance":46406.98,"reference_id":"498320351284","date_time":"21/10/26","category":"banking","spam":false}

=== Sample 2 ===
System:    Extract structured JSON from SMS. Output valid JSON only.
User SMS:  FLAT 70% OFF + Extra 15% cashback on top fashion brands! Limited period festive sale on Myntra. Use code EXTRA15 at checkout. Shop now: http://bit.ly/instant-loan25
Assistant: {"type":"promo","currency":"INR","merchant":"Myntra","category":"shopping","spam":true}

=== Sample 3 ===
System:    Extract structured JSON from SMS. Output valid JSON only.
User SMS:  [AD-HDFCBK] 546688 is your OTP for online purchase of Rs.33,

## 3. Base Model Inspection & Zero-Shot Baseline
Run `SMSExtractorEngine` on the **base model** (before fine-tuning) and inspect the chat template.  
If running offline without cached Hugging Face weights, `SMSExtractorEngine` cleanly drops into regex-fallback mode (`fallback_used=True`).

In [4]:
from slm_sms.inference.engine import SMSExtractorEngine

# Toggle True to load the base model for a zero-shot baseline comparison
RUN_ZERO_SHOT_BASELINE = True

test_sms = (
    "[VM-HDFCBK] Rs.1,450.00 debited from A/c **4821 on 23-11-26 "
    "to swiggy@axisbank (UPI Ref: 432819284129). Avl Bal: Rs.34,210.50."
)

if RUN_ZERO_SHOT_BASELINE:
    base_engine = SMSExtractorEngine(
        model_id_or_path=MODEL_ID,
        model_revision=MODEL_REVISION,
        max_new_tokens=settings.max_new_tokens,
    )
    if base_engine.tokenizer is not None:
        prompt_str = base_engine.tokenizer.apply_chat_template(
            base_engine.build_prompt_messages(test_sms),
            tokenize=False,
            add_generation_prompt=True,
        )
        print("Formatted Qwen chat prompt:\n", prompt_str)

    baseline_result = base_engine.extract(test_sms)
    print("Baseline extraction result:")
    print(json.dumps(baseline_result, indent=2))

    # Free the base model before training so MPS/CUDA doesn't hold 2 model copies
    del base_engine
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

{"timestamp": "2026-11-25T20:12:09", "level": "INFO", "logger": "slm_sms.inference.engine", "message": "Loading inference engine", "trace_id": "531e1f788c596abf40e9f7508a460e36", "span_id": "264ce82b311a759c", "trace_flags": "03", "base_model": "Qwen/Qwen3.5-2B", "revision": null, "adapter": null}
{"timestamp": "2026-11-25T20:12:09", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-2B \"HTTP/1.1 200 OK\"", "trace_id": "531e1f788c596abf40e9f7508a460e36", "span_id": "264ce82b311a759c", "trace_flags": "03"}
{"timestamp": "2026-11-25T20:12:09", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-2B/tree/main/additional_chat_templates?recursive=false&expand=false \"HTTP/1.1 404 Not Found\"", "trace_id": "531e1f788c596abf40e9f7508a460e36", "span_id": "264ce82b311a759c", "trace_flags": "03"}
{"timestamp": "2026-11-25T20:12:09", "level": "INFO", "logger": "httpx2", "me

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

{"timestamp": "2026-11-25T20:12:12", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/custom_generate/generate.py \"HTTP/1.1 404 Not Found\"", "trace_id": "531e1f788c596abf40e9f7508a460e36", "span_id": "264ce82b311a759c", "trace_flags": "03"}
{"timestamp": "2026-11-25T20:12:13", "level": "INFO", "logger": "slm_sms.inference.engine", "message": "Inference engine ready", "trace_id": "531e1f788c596abf40e9f7508a460e36", "span_id": "264ce82b311a759c", "trace_flags": "03", "device": "mps", "dtype": "torch.float16"}


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Formatted Qwen chat prompt:
 <|im_start|>system
Extract structured JSON from SMS. Output valid JSON only.<|im_end|>
<|im_start|>user
[VM-HDFCBK] Rs.1,450.00 debited from A/c **4821 on 23-11-26 to swiggy@axisbank (UPI Ref: 432819284129). Avl Bal: Rs.34,210.50.<|im_end|>
<|im_start|>assistant
<think>

</think>


{"timestamp": "2026-11-25T20:12:17", "level": "WARNING", "logger": "slm_sms.inference.engine", "message": "Failed to parse LLM JSON — using regex fallback", "trace_id": "d79375e4c8384976039e7529f3fa830c", "span_id": "51fb74629a51f7eb", "trace_flags": "03", "error": "1 validation error for SMSEntity\ntype\n  Field required [type=missing, input_value={'amount': 1450.0, 'curre...lable_balance': 34210.5}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.14/v/missing"}
Baseline extraction result:
{
  "entity": {
    "type": "debit",
    "amount": 1450.0,
    "currency": "INR",
    "account": "4821",
    "spam": false
  },
  "latency_ms": 3845.97,
  "fa

## 4. Setup PEFT LoRA (Low-Rank Adaptation)
Pull LoRA hyperparameters directly from `config/train_config.yaml` (`r`, `lora_alpha`, `lora_dropout`, and all Qwen attention + MLP projections).

In [5]:
from slm_sms.models.model_loader import get_lora_config

lora_cfg_dict = train_cfg["lora"]
lora_config = get_lora_config(
    r=int(lora_cfg_dict["r"]),
    lora_alpha=int(lora_cfg_dict["lora_alpha"]),
    lora_dropout=float(lora_cfg_dict["lora_dropout"]),
    target_modules=lora_cfg_dict.get("target_modules"),
)
print("LoRA Configuration:")
print(lora_config)

LoRA Configuration:
LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.22.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'q_proj', 'v_proj', 'up_proj', 'down_proj', 'o_proj', 'k_proj', 'gate_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, astra_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlor

## 5. Fine-Tuning with `SMSTrainingPipeline` (SFTTrainer + PEFT + OTel Spans)
Executes the exact same `SMSTrainingPipeline` used by `slm-sms train` / `make train-model`.  
Set `RUN_TRAINING = True` (default) when running on a CUDA GPU or Apple Silicon MPS machine with Hugging Face access / cached weights.

In [6]:
from slm_sms.data.dataset import load_hf_dataset_from_dir
from slm_sms.training.trainer import SMSTrainingPipeline

RUN_TRAINING = True  # Set False to skip training and test an existing adapter in OUTPUT_DIR

t_cfg = train_cfg["training"]
m_cfg = train_cfg["model"]
hf_dataset = load_hf_dataset_from_dir(DATA_DIR)

pipeline = SMSTrainingPipeline(
    model_id=MODEL_ID,
    output_dir=OUTPUT_DIR,
    lora_r=int(lora_cfg_dict["r"]),
    lora_alpha=int(lora_cfg_dict["lora_alpha"]),
    lora_dropout=float(lora_cfg_dict["lora_dropout"]),
    use_4bit=bool(m_cfg.get("use_4bit", False)),
    use_8bit=bool(m_cfg.get("use_8bit", False)),
)

if RUN_TRAINING:
    # Ensure any previously loaded model objects are cleared before fine-tuning
    for _var in ("eval_engine", "base_engine"):
        globals().pop(_var, None)
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()

    train_metrics = pipeline.train(
        dataset=hf_dataset,
        num_train_epochs=int(t_cfg["num_train_epochs"]),
        per_device_train_batch_size=int(t_cfg["per_device_train_batch_size"]),
        per_device_eval_batch_size=t_cfg.get("per_device_eval_batch_size"),
        gradient_accumulation_steps=int(t_cfg["gradient_accumulation_steps"]),
        learning_rate=float(t_cfg["learning_rate"]),
        max_length=int(t_cfg["max_length"]),
        warmup_ratio=float(t_cfg["warmup_ratio"]),
        logging_steps=int(t_cfg["logging_steps"]),
        eval_steps=t_cfg.get("eval_steps", 50),
        save_total_limit=int(t_cfg["save_total_limit"]),
        gradient_checkpointing=t_cfg.get("gradient_checkpointing"),
    )
    print("Training & evaluation metrics:")
    print(json.dumps(train_metrics, indent=2))
else:
    print(f"Skipping training — will load adapter from {OUTPUT_DIR} if present.")

{"timestamp": "2026-11-25T20:12:17", "level": "INFO", "logger": "slm_sms.training.trainer", "message": "Initializing training model", "trace_id": "ab0f906aa766577a8e0257567b7c1487", "span_id": "8450bf32651cbdcf", "trace_flags": "03", "model_id": "Qwen/Qwen3.5-2B"}
{"timestamp": "2026-11-25T20:12:17", "level": "INFO", "logger": "slm_sms.models.model_loader", "message": "Loading tokenizer", "trace_id": "ab0f906aa766577a8e0257567b7c1487", "span_id": "8450bf32651cbdcf", "trace_flags": "03", "model_id": "Qwen/Qwen3.5-2B", "revision": null}
{"timestamp": "2026-11-25T20:12:17", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-2B \"HTTP/1.1 200 OK\"", "trace_id": "ab0f906aa766577a8e0257567b7c1487", "span_id": "8450bf32651cbdcf", "trace_flags": "03"}
{"timestamp": "2026-11-25T20:12:17", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-2B/tree/main/additional_chat_tem

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

{"timestamp": "2026-11-25T20:12:18", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/custom_generate/generate.py \"HTTP/1.1 404 Not Found\"", "trace_id": "ab0f906aa766577a8e0257567b7c1487", "span_id": "8450bf32651cbdcf", "trace_flags": "03"}
{"timestamp": "2026-11-25T20:12:20", "level": "INFO", "logger": "slm_sms.models.model_loader", "message": "Trainable params: 10,911,744 || All params: 1,892,736,832 || Trainable %: 0.577%", "trace_id": "ab0f906aa766577a8e0257567b7c1487", "span_id": "8450bf32651cbdcf", "trace_flags": "03"}
{"timestamp": "2026-11-25T20:12:20", "level": "INFO", "logger": "slm_sms.training.trainer", "message": "Configured SFT Trainer", "trace_id": "ab0f906aa766577a8e0257567b7c1487", "span_id": "432c404746bb8034", "trace_flags": "03", "device": "mps", "use_cpu": false, "fp16": false, "bf16": false, "eval_strategy": "steps"}


Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

... (14 outputs left out)

{"timestamp": "2026-11-25T20:39:59", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/config.json \"HTTP/1.1 307 Temporary Redirect\"", "trace_id": "ab0f906aa766577a8e0257567b7c1487", "span_id": "1c80317fa3b1799d", "trace_flags": "03"}
{"timestamp": "2026-11-25T20:39:59", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-2B/15852e8c16360a2fea060d615a32b45270f8a8fc/config.json?%2FQwen%2FQwen3.5-2B%2Fresolve%2Fmain%2Fconfig.json=&etag=%22d30a15be1bbfd610f58ad80f1fbeb6778f58a80f%22 \"HTTP/1.1 200 OK\"", "trace_id": "ab0f906aa766577a8e0257567b7c1487", "span_id": "1c80317fa3b1799d", "trace_flags": "03"}
{"timestamp": "2026-11-25T20:39:59", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-2B/revision/main \"HTTP/1.1 200 OK\"", "trace_id": "ab0f906aa766577a8e0257567b7c1487", "span_

/Users/parthsojitra/Work/Projects/SLMTraining/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
0.432991,0.404358,125,0.420548,0.848072,337466.000000


***** eval metrics *****
  eval_entropy             =   0.4205
  eval_loss                =   0.4044
  eval_mean_token_accuracy =   0.8481
  eval_num_tokens          = 337466.0
Training & evaluation metrics:
{
  "train_runtime": 1657.1601,
  "train_samples_per_second": 1.207,
  "train_steps_per_second": 0.075,
  "total_flos": 2802677008595712.0,
  "train_loss": 0.6060955982208251,
  "epoch": 1.0,
  "eval_loss": 0.40435826778411865,
  "eval_entropy": 0.42054837636152903,
  "eval_mean_token_accuracy": 0.8480719202359518,
  "eval_num_tokens": 337466.0
}


## 6. Test-Set Evaluation & Benchmarking
Runs [`SMSExtractorEngine`](../src/slm_sms/inference/engine.py) with the trained LoRA adapter over `data/test.jsonl` (using native GPU batch inference via `extract_batch`) and scores predictions with `compute_extraction_metrics()`—mirroring `slm-sms evaluate` and `scripts/evaluate.py`.

In [7]:
from slm_sms.data.dataset import load_jsonl
from slm_sms.training.metrics import compute_extraction_metrics

# Free training pipeline weights/optimizer states before loading eval inference engine
globals().pop("pipeline", None)
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
elif torch.cuda.is_available():
    torch.cuda.empty_cache()

MAX_EVAL_SAMPLES = 50
test_records = load_jsonl(DATA_DIR / "test.jsonl")[:MAX_EVAL_SAMPLES]

adapter_dir = str(OUTPUT_DIR) if OUTPUT_DIR.exists() else None
eval_engine = SMSExtractorEngine(
    model_id_or_path=MODEL_ID,
    adapter_path=adapter_dir,
    model_revision=MODEL_REVISION,
    max_new_tokens=settings.max_new_tokens,
)

sms_inputs = [rec["messages"][1]["content"] for rec in test_records]
ground_truths = [json.loads(rec["messages"][2]["content"]) for rec in test_records]

# Run batched extraction (automatically uses GPU batching when model is loaded)
batch_results = eval_engine.extract_batch(sms_inputs)
predictions = [json.dumps(r["entity"]) for r in batch_results]
fallback_count = sum(1 for r in batch_results if r["fallback_used"])

report = compute_extraction_metrics(predictions, ground_truths)
print(f"Evaluated {len(test_records)} samples (adapter={adapter_dir}, fallbacks={fallback_count})")
print("\nExtraction Benchmark Report:")
for metric_name, val in report.to_dict().items():
    print(f"  {metric_name:22s}: {val}")

{"timestamp": "2026-11-25T20:41:07", "level": "INFO", "logger": "slm_sms.inference.engine", "message": "Loading inference engine", "trace_id": "23b8c1e9392456de3eb13b9046685257", "span_id": "1a3d1fa7bc8960a9", "trace_flags": "03", "base_model": "Qwen/Qwen3.5-2B", "revision": null, "adapter": "/Users/parthsojitra/Work/Projects/SLMTraining/output/qwen-sms-lora-notebook"}
{"timestamp": "2026-11-25T20:41:08", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3.5-2B \"HTTP/1.1 200 OK\"", "trace_id": "23b8c1e9392456de3eb13b9046685257", "span_id": "1a3d1fa7bc8960a9", "trace_flags": "03"}


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

{"timestamp": "2026-11-25T20:41:10", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-2B/resolve/main/custom_generate/generate.py \"HTTP/1.1 404 Not Found\"", "trace_id": "23b8c1e9392456de3eb13b9046685257", "span_id": "1a3d1fa7bc8960a9", "trace_flags": "03"}
{"timestamp": "2026-11-25T20:41:10", "level": "INFO", "logger": "slm_sms.inference.engine", "message": "Loading LoRA adapter", "trace_id": "23b8c1e9392456de3eb13b9046685257", "span_id": "1a3d1fa7bc8960a9", "trace_flags": "03", "adapter_path": "/Users/parthsojitra/Work/Projects/SLMTraining/output/qwen-sms-lora-notebook"}
{"timestamp": "2026-11-25T20:41:11", "level": "INFO", "logger": "slm_sms.inference.engine", "message": "Inference engine ready", "trace_id": "23b8c1e9392456de3eb13b9046685257", "span_id": "1a3d1fa7bc8960a9", "trace_flags": "03", "device": "mps", "dtype": "torch.float16"}
Evaluated 50 samples (adapter=/Users/parthsojitra/Work/Projects/SLMTraining/output/qwen-sms-

## 7. Interactive Single & Batched Inference (`SMSExtractorEngine`)
Test both `extract()` and `extract_batch()` on diverse Indian SMS messages (UPI debits, OTPs, SBI salary credits, lakh-format amounts, and spam/phishing).

In [8]:
sample_queries = [
    "[VM-HDFCBK] Rs.1,450.00 debited from A/c **4821 on 23-11-26 to swiggy@axisbank (UPI Ref: 432819284129). Avl Bal: Rs.34,210.50.",
    "489201 is your secret OTP for transaction of Rs.2,499.00 on your Axis Bank Card **7712 at ZOMATO. Valid for 5 mins.",
    "[BZ-SBIINB] A/c *8921 credited by Rs.1,45,000.00 on 23-Nov-26 by Salary transfer from TECH CORP PVT LTD. Avl Bal: Rs.2,87,400.00.",
    "Congratulations! Pre-approved personal loan of Rs.5,00,000 without income proof. Click bit.ly/loan-instant to claim before midnight!",
]

# Single inference example
single_res = eval_engine.extract(sample_queries[0])
print("=== Single Inference ===")
print(json.dumps(single_res, indent=2))

# Batched inference across all 4 queries
print("\n=== Batch Inference ===")
for sms, res in zip(sample_queries, eval_engine.extract_batch(sample_queries)):
    print(f"\nInput SMS: {sms}")
    print(f"Latency: {res['latency_ms']} ms | Fallback used: {res['fallback_used']}")
    print(json.dumps(res["entity"], indent=2))

=== Single Inference ===
{
  "entity": {
    "type": "debit",
    "amount": 1450.0,
    "currency": "INR",
    "merchant": "swiggy@axisbank",
    "account": "4821",
    "balance": 34210.5,
    "reference_id": "432819284129",
    "date_time": "23-11-26 11:09",
    "category": "food_dining",
    "spam": false
  },
  "latency_ms": 3616.75,
  "fallback_used": false
}

=== Batch Inference ===

Input SMS: [VM-HDFCBK] Rs.1,450.00 debited from A/c **4821 on 23-11-26 to swiggy@axisbank (UPI Ref: 432819284129). Avl Bal: Rs.34,210.50.
Latency: 1881.88 ms | Fallback used: False
{
  "type": "debit",
  "amount": 1450.0,
  "currency": "INR",
  "merchant": "swiggy@axisbank",
  "account": "4821",
  "balance": 34210.5,
  "reference_id": "432819284129",
  "date_time": "23-11-26 11:59:09",
  "category": "food_dining",
  "spam": false
}

Input SMS: 489201 is your secret OTP for transaction of Rs.2,499.00 on your Axis Bank Card **7712 at ZOMATO. Valid for 5 mins.
Latency: 1881.88 ms | Fallback used: False
{

## 8. FastAPI Microservice & OpenTelemetry Trace Verification
Exercise the production [`create_app()`](../src/slm_sms/inference/server.py) ASGI app in-process with `TestClient` and an OpenTelemetry `InMemorySpanExporter`, verifying:
- `/health/live` & `/health/ready` Kubernetes probes
- `/api/v1/extract` & `/api/v1/extract/batch` HTTP contracts and validation
- `X-Request-ID` & `X-Trace-ID` response headers + parent-child span linkage across the `ThreadPoolExecutor` boundary

In [9]:
from fastapi.testclient import TestClient
from opentelemetry.sdk.metrics.export import InMemoryMetricReader
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from slm_sms.core.telemetry import shutdown_telemetry
from slm_sms.inference.server import create_app

os.environ["SLM_LAZY_LOAD"] = "true"
os.environ["SLM_DRAIN_SECONDS"] = "0"

span_exporter = InMemorySpanExporter()
metric_reader = InMemoryMetricReader()
configure_telemetry(
    service_name="slm-sms-notebook",
    span_exporter=span_exporter,
    metric_reader=metric_reader,
    force=True,
)

try:
    notebook_app = create_app(init_telemetry=False)
    with TestClient(notebook_app) as client:
        # Reuse our already initialized eval_engine inside app.state
        client.app.state.engine = eval_engine

        print("Liveness: ", client.get("/health/live").json())
        print("Readiness:", client.get("/health/ready").json())

        resp = client.post(
            "/api/v1/extract",
            json={"text": sample_queries[1]},
            headers={"X-Request-ID": "nb-demo-001"},
        )
        print("\nPOST /api/v1/extract status:", resp.status_code)
        print("X-Request-ID:", resp.headers.get("X-Request-ID"))
        print("X-Trace-ID:  ", resp.headers.get("X-Trace-ID"))
        print("Body:", json.dumps(resp.json(), indent=2))

    emitted_spans = [s.name for s in span_exporter.get_finished_spans()]
    print("\nCaptured OpenTelemetry spans:", emitted_spans)
finally:
    shutdown_telemetry()

{"timestamp": "2026-11-25T20:53:09", "level": "INFO", "logger": "slm_sms.core.telemetry", "message": "OpenTelemetry configured", "otel_service_name": "slm-sms-notebook", "otel_exporter": "none", "otel_protocol": null, "otel_sample_rate": 1.0}
{"timestamp": "2026-11-25T20:53:09", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: GET http://testserver/health/live \"HTTP/1.1 200 OK\""}
Liveness:  {'status': 'alive'}
{"timestamp": "2026-11-25T20:53:09", "level": "INFO", "logger": "httpx2", "message": "HTTP Request: GET http://testserver/health/ready \"HTTP/1.1 200 OK\""}
Readiness: {'status': 'ready', 'device': 'mps'}
{"timestamp": "2026-11-25T20:53:09", "level": "INFO", "logger": "slm_sms.inference.server", "message": "extract_start", "trace_id": "c241330b01a9e71fde8a774bcf36d58b", "span_id": "28df6ec4ce4a2bbd", "trace_flags": "03", "request_id": "nb-demo-001", "text_length": 115}
{"timestamp": "2026-11-25T20:53:11", "level": "INFO", "logger": "slm_sms.inference.server", "mes

## 9. Export Merged Model & Production Deployment
Merge the LoRA adapter into standalone Hugging Face `safetensors` weights for **vLLM**, **Triton**, **Ollama**, or **Gunicorn + FastAPI**:

```bash
# Merge LoRA weights into base model
make export-model

# Launch production Gunicorn + UvicornWorker server
make serve-prod

# Or run with OpenTelemetry OTLP export (e.g. Jaeger / Grafana Collector at http://localhost:4318)
make serve-otel
```
